In [ ]:
# project_name = 'M000'
# working_dir = f'{project_name}'
# models_dir = f'{working_dir}/models'
# dataset_dir = f'datasets/'
# model_name = f"M0000_E03"

# encoder_checkpoint = f'{models_dir}/2024-07-18-M0000-encoder-loss_0.157.pt'
# transformer_checkpoint = f'{models_dir}/2024-07-20_M0000.E11-transformer.ckpt.epoch_2_avg_loss_0.770.pt'
# dataset_path = f"{dataset_dir}/objverse_shapenet_modelnet_max_250faces_186M_tokens_cpu.npz"

In [2]:
from pathlib import Path 
import gc     
import os
from meshgpt_pytorch import MeshDataset
dataset_path = f"datasets/objverse_shapenet_modelnet_max_250faces_186M_tokens_cpu.npz"
dataset = MeshDataset.load(dataset_path)

[MeshDataset] Loaded 218835 entries
[MeshDataset] Created from 218835 entries


In [ ]:
# labels = list(set(item["texts"] for item in dataset.data))
# labels

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Count the occurrences of each label (converted to lowercase)
label_counts = Counter(item["texts"].lower() for item in dataset.data)

# Get labels and their counts
labels, counts = zip(*label_counts.items())

# Sort labels by counts in descending order
sorted_labels, sorted_counts = zip(*sorted(label_counts.items(), key=lambda x: x[1], reverse=True))

# Filter to show only the top 30 labels
top_n = 30
top_labels = sorted_labels[:top_n]
top_counts = sorted_counts[:top_n]

# Plot the label distribution
plt.figure(figsize=(12, 10))  # Increased figure size for better readability
plt.barh(top_labels, top_counts, color='skyblue')
plt.xlabel('Number of Samples')
plt.ylabel('Labels')
plt.title(f'Top {top_n} Label Distribution in Dataset')
plt.gca().invert_yaxis()
plt.xticks(rotation=45)  # Rotate x-axis labels if necessary
plt.show()


In [ ]:
top_labels[:10]

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Convert labels to lowercase and find unique labels
labels = list(set(item["texts"].lower() for item in dataset.data))

# Sort labels by their length in descending order
sorted_labels_by_length = sorted(labels, key=len, reverse=True)

# Get the top 50 labels by length
top_50_labels_by_length = sorted_labels_by_length[:50]

# Count the occurrences of the top 50 labels in the dataset
label_counts = Counter(item["texts"].lower() for item in dataset.data)
top_50_counts = [label_counts[label] for label in top_50_labels_by_length]

# Plot the distribution of the top 50 labels by length
plt.figure(figsize=(12, 10))
plt.barh(top_50_labels_by_length, top_50_counts, color='skyblue')
plt.xlabel('Number of Samples')
plt.ylabel('Labels')
plt.title('Top 50 Labels by Length in Dataset')
plt.gca().invert_yaxis()
plt.xticks(rotation=45)
plt.show()

# List the top 50 labels
print("Top 50 Labels by Length:")
for label in top_50_labels_by_length:
    print(label)


In [12]:
chairs = set()
for r, item in enumerate(dataset.data): 
    if 'chair' in item['texts'].lower():
        chairs.add(item['texts'].lower())

print(len(chairs))
display(chairs)

123


{'#30days3d - day 15 - chair',
 '14 armchair #householdpropschallenge',
 'a table with a chair',
 'abiola0211 chair03blend',
 'aggie stadium chair',
 'akingbade0211 chair3',
 'amparan0211chair03blend',
 'anna ciabattoni-steltman chair',
 'armchair',
 'armchair chair',
 'armchair: household props 14',
 'baker0210 chair02',
 'baker0211 chair03',
 "baldi's basics chair",
 'basic chair',
 'blanton0210 chair02',
 'blanton0211 chair03',
 'boston02011 chair03',
 'briscoe0209 chair03',
 'calvo0211 chair03',
 'cantilever chair armchair chair',
 'chair',
 'chair - low poly',
 'chair 02 - mythical beasts jousting assets',
 'chair 2',
 'chair 2.1',
 'chair armchair',
 'chair blockbench',
 'chair fbx',
 'chair from poly by google',
 'chair large',
 'chair medium',
 'chair model',
 'chair wood',
 'chair | admin | among us',
 'chair 🪑',
 'chair-1',
 'chair1',
 'chaise longue chaise daybed chair',
 'chavez 02010 chair02',
 'claiborne0211 chair03',
 'club chair chair',
 'contreras0211 chair03',
 'david

In [15]:
from pathlib import Path
import numpy as np
 
folder = f"'datasets/objverse_shapenet_modelnet_max_250faces_186M_tokens/"
obj_file_path = Path(folder)
obj_file_path.mkdir(exist_ok = True, parents = True)
   
all_vertices = []
all_faces = []
vertex_offset = 0
translation_distance = 0.5

query = ['chair']
count = 0

# Configurable grid size
num_columns = 20  # Number of columns per layer
num_rows = num_columns # Number of rows per layer

index = []

for r, item in enumerate(dataset.data): 

    if 'chair' not in item['texts'].lower():    
        continue

    key = f'{item['texts'].lower()}_{len(item['vertices'])}'
    
    if key in index:
        #print('Skipping duplicate')
        continue

    print(f'Matched {key}')
        
    index.append(key)
               
    col = count % num_columns
    row = (count // num_columns) % num_rows
    layer = count // (num_columns * num_rows)

    #print(f"c:{count} row:{row} col:{col} layer:{layer}")

    vertices_copy =  np.copy(item['vertices'])    
    vertices_copy[:, 2] += translation_distance * (col / 0.2 - 1) # Translate Y
    vertices_copy[:, 0] += translation_distance * (row / 0.2 - 1) # Translate X
    vertices_copy[:, 1] += translation_distance * (layer / 0.2 - 1) # Translate Z
    
    for vert in vertices_copy:
        vertex = vert#.to('cpu')
        all_vertices.append(f"v {float(vertex[0])}  {float(vertex[1])}  {float(vertex[2])}\n") 
    for face in item['faces']:
        all_faces.append(f"f {face[0]+1+ vertex_offset} {face[1]+ 1+vertex_offset} {face[2]+ 1+vertex_offset}\n")  
    vertex_offset = len(all_vertices)

    #if count > 20:
    #    break
        
    count += 1

print(f'Matched {count} items')
obj_file_content = "".join(all_vertices) + "".join(all_faces)
 
obj_file_path = f'{folder}/3d_models_inspect.obj' 
with open(obj_file_path, "w") as file:
    file.write(obj_file_content)  

Matched wooden chair_126
Matched small chair_126
Matched williams0211 chair03_72
Matched chair_64
Matched fritz0209chair_72
Matched sanchez0211 chair03_114
Matched chair_125
Matched simple 3d chair_72
Matched amparan0211chair03blend_60
Matched low poly wooden chair_80
Matched blanton0211 chair03_86
Matched ye0211 chair03_102
Matched wooden chair_104
Matched etheridge0211 chair03_84
Matched 14 armchair #householdpropschallenge_128
Matched chair_84
Matched #30days3d - day 15 - chair_87
Matched fiscochair_36
Matched wooden chair_66
Matched chair_70
Matched simple chair_96
Matched literally a chair_40
Matched chair_116
Matched chair_78
Matched chair_88
Matched mari chair_104
Matched hubbard0211 chair03_114
Matched akingbade0211 chair3_114
Matched chair_72
Matched chair_93
Matched chair_79
Matched martinez0209 chair02_28
Matched long chair_112
Matched low poly chair_48
Matched chair_80
Matched chair 2.1_44
Matched chair 🪑_90
Matched chair blockbench_82
Matched stewart0211 chair03_114
Matche